# KcBERT + Embedding + LLM 기반 음식점 리뷰 분석 AI Tool

이 코드는 기존 “완전 처음 코드”의 서비스 흐름을 유지하되, 팀 합의사항을 반영하여 **LLM이 모든 1차 분류를 담당하지 않도록** 수정한 버전입니다.

핵심 구조:

1. 엑셀/CSV에서 특정 음식점 리뷰 불러오기  
2. KR3 데이터셋으로 KcBERT 감성분류 모델 fine-tuning  
3. 학습된 KcBERT로 수집 리뷰의 긍정/부정/중립 1차 분류  
4. 긍정 리뷰와 부정/주의 리뷰 분리  
5. 각 그룹을 OpenAI embedding으로 변환  
6. 긍정 리뷰끼리 클러스터링  
7. 부정/주의 리뷰끼리 클러스터링  
8. LLM이 각 클러스터 의미 해석  
9. 메뉴별 반응과 최근 리뷰 가중치 반영  
10. LLM이 최종 음식점 AI 프로필 생성  

발표 포인트:

- KcBERT: 반복적이고 안정적인 1차 감성분류 담당  
- Embedding + KMeans: 비슷한 의미의 리뷰 그룹화 담당  
- LLM: 클러스터 해석과 최종 사용자용 프로필 생성 담당  


In [1]:
# ============================================================
# 0. 패키지 설치
# ============================================================
!pip install -q pandas numpy scikit-learn openpyxl tqdm datasets transformers accelerate torch openai

In [7]:
# ============================================================
# 1. 기본 설정
# ============================================================
import os
import re
import json
import time
import math
import numpy as np
import pandas as pd

from tqdm import tqdm
from collections import Counter, defaultdict

import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_distances

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from openai import OpenAI
# from google.colab import userdata
from dotenv import load_dotenv
import os

# ------------------------------------------------------------
# OpenAI API Key (Colab Secrets에서 가져오기)
# ------------------------------------------------------------
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ------------------------------------------------------------
# 파일 설정
# ------------------------------------------------------------
DATA_PATH = "place_reviews_liked.csv"   # csv/xlsx 모두 가능
TARGET_STORE = "피자스쿨 수원성대점"     # 데이터에 존재하는 식당으로 수정

# ------------------------------------------------------------
# KcBERT 학습 설정
# ------------------------------------------------------------
TRAIN_KCBERT = True                    # 이미 학습된 모델이 있으면 False로 바꾸고 MODEL_SAVE_DIR 사용
MODEL_NAME = "beomi/kcbert-base"
MODEL_SAVE_DIR = "./kcbert_kr3_sentiment_model"

MAX_PER_CLASS = 2000                   # KR3에서 긍정/부정 각각 최대 사용 개수
MAX_LENGTH = 96
NUM_EPOCHS = 1
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
RANDOM_SEED = 42

# ------------------------------------------------------------
# OpenAI 모델 설정
# ------------------------------------------------------------
LLM_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ------------------------------------------------------------
# 클러스터링 설정
# ------------------------------------------------------------
MIN_CLUSTER_SIZE = 4
MAX_CLUSTERS = 5
REPRESENTATIVE_REVIEWS_PER_CLUSTER = 5

# ------------------------------------------------------------
# 컬럼명 후보
# ------------------------------------------------------------
RESTAURANT_COL_CANDIDATES = ["restaurant_name", "restaurant", "store_name", "식당명", "음식점명", "가게명", "상호명"]
REVIEW_COL_CANDIDATES     = ["review_text", "review", "리뷰내용", "리뷰", "내용", "본문"]
MENU_COL_CANDIDATES       = ["menu", "주문메뉴", "메뉴", "ordered_menu", "주문 메뉴"]
DATE_COL_CANDIDATES       = ["date", "review_date", "작성일", "날짜", "리뷰날짜", "방문일"]

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"설정 완료: 대상 식당 - {TARGET_STORE}")

설정 완료: 대상 식당 - 피자스쿨 수원성대점


In [10]:
# # from google.colab import files
# uploaded = files.upload()

# import os
# print(os.listdir())

In [11]:
# ============================================================
# 2. 리뷰 데이터 로드 및 전처리 (수정됨)
# ============================================================

def find_col(df, candidates, required=False, name="column"):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"{name} 컬럼을 찾지 못했습니다. 후보: {candidates}\n현재 컬럼: {list(df.columns)}")
    return None


def load_review_data(path):
    if path.lower().endswith(".csv"):
        df_raw = pd.read_csv(path)
    elif path.lower().endswith((".xlsx", ".xls")):
        df_raw = pd.read_excel(path)
    else:
        raise ValueError("지원 파일 형식은 csv, xlsx, xls입니다.")

    # 추가된 후보: 'visit_date'를 DATE_COL_CANDIDATES에 명시적으로 체크
    res_col = find_col(df_raw, RESTAURANT_COL_CANDIDATES, required=False, name="restaurant")
    rev_col = find_col(df_raw, REVIEW_COL_CANDIDATES, required=True, name="review")
    menu_col = find_col(df_raw, MENU_COL_CANDIDATES, required=False, name="menu")
    date_col = find_col(df_raw, DATE_COL_CANDIDATES + ["visit_date"], required=False, name="date")

    df = pd.DataFrame()
    df["restaurant_name"] = df_raw[res_col].astype(str).str.strip() if res_col else "음식점"
    df["review_text"] = df_raw[rev_col].astype(str).str.strip()
    df["menu_raw"] = df_raw[menu_col].astype(str).str.strip() if menu_col else ""

    if date_col:
        # '2026.04.08.' 같은 형식을 처리하기 위해 숫자가 아닌 마지막 점 제거
        clean_date = df_raw[date_col].astype(str).str.replace(r'[^0-9.]', '', regex=True).str.rstrip('.')
        df["review_date"] = pd.to_datetime(clean_date, format='%Y.%m.%d', errors="coerce")
    else:
        df["review_date"] = pd.NaT

    df = df[df["review_text"].notna()]
    df = df[df["review_text"].str.len() > 1]
    df = df.drop_duplicates(subset=["restaurant_name", "review_text"]).reset_index(drop=True)

    if TARGET_STORE:
        df = df[df["restaurant_name"].astype(str).str.contains(TARGET_STORE, na=False)].reset_index(drop=True)

    return df

df = load_review_data(DATA_PATH)
print(f"로드된 리뷰 수: {len(df)}")
if len(df) > 0:
    display(df.head())
else:
    print("조건에 맞는 리뷰가 없습니다.")

로드된 리뷰 수: 0
조건에 맞는 리뷰가 없습니다.


In [6]:
# 업로드된 파일의 실제 컬럼과 상위 데이터 확인
import pandas as pd

try:
    # 가장 최근에 업로드된 파일명을 확인하거나 기본 DATA_PATH를 읽음
    temp_df = pd.read_csv(DATA_PATH)
    print(f"--- 파일 '{DATA_PATH}' 정보 ---")
    print(f"전체 행 수: {len(temp_df)}")
    print(f"컬럼명: {list(temp_df.columns)}")

    # 식당명 컬럼 확인
    res_col = find_col(temp_df, RESTAURANT_COL_CANDIDATES)
    if res_col:
        unique_stores = temp_df[res_col].unique()
        print(f"존재하는 식당 목록 (일부): {unique_stores[:10]}")
        print(f"'{TARGET_STORE}' 포함 여부: {TARGET_STORE in unique_stores}")
    else:
        print("식당명 컬럼을 찾을 수 없습니다.")

    display(temp_df.head())
except Exception as e:
    print(f"에러 발생: {e}")

에러 발생: [Errno 2] No such file or directory: 'kakao_reviews_new.csv'


In [13]:
# ============================================================
# 3. 최근 리뷰 가중치 계산
# ============================================================

def compute_recency_weight(date_series):
    """
    최근 리뷰일수록 더 큰 가중치 부여.
    날짜가 전부 없으면 모든 리뷰 weight = 1.0
    """
    if date_series.isna().all():
        return pd.Series([1.0] * len(date_series), index=date_series.index)

    max_date = date_series.max()
    days = (max_date - date_series).dt.days

    weights = []
    for d in days:
        if pd.isna(d):
            weights.append(1.0)
        elif d <= 30:
            weights.append(1.5)
        elif d <= 90:
            weights.append(1.2)
        else:
            weights.append(1.0)

    return pd.Series(weights, index=date_series.index)


df["recency_weight"] = compute_recency_weight(df["review_date"])

print(df[["review_date", "recency_weight"]].head())
print(df["recency_weight"].value_counts(dropna=False))

  review_date  recency_weight
0  2026-05-08             1.5
1  2025-12-18             1.0
2  2025-12-17             1.0
3  2025-06-05             1.0
4  2025-04-08             1.0
recency_weight
1.0    11
1.5     1
Name: count, dtype: int64


In [32]:
import re
import pandas as pd

def normalize_menu_string(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    s = str(x).strip()
    if not s or s.lower() in ["nan", "none"]:
        return []
    parts = re.split(r"[,/\n+|·]", s)
    return [p.strip() for p in parts if p.strip()]

def extract_menus_from_text(text, menu_raw):
    menus = normalize_menu_string(menu_raw)
    menu_keywords = ['피자', '까르보나라', '스파게티', '오븐스파게티', '치즈피자', '페퍼로니', '고구마피자', '포테이토', '불고기피자', '도이치바이트']
    found_in_text = [kw for kw in menu_keywords if kw in str(text)]
    combined = menus + found_in_text
    stop_words = {"메뉴", "음식", "주문", "없음", "nan", "none"}
    cleaned = [re.sub(r"\s+", " ", str(m)).strip() for m in combined if len(str(m)) > 1 and str(m).lower() not in stop_words]
    return sorted(set(cleaned))

df["menu_candidates"] = df.apply(lambda row: extract_menus_from_text(row["review_text"], row["menu_raw"]), axis=1)
print(f"메뉴 추출 완료. 추출된 리뷰 수: {len(df[df['menu_candidates'].map(len) > 0])} / {len(df)}")

메뉴 추출 완료. 추출된 리뷰 수: 7 / 12


## 5. KR3 데이터셋으로 KcBERT 감성분류 모델 학습

이 단계는 팀 합의사항인 “1차 긍/부정 분류는 BERT 계열 경량 모델로 따로 두자”를 반영한 부분입니다.

- 학습 데이터: KR3 한국어 식당 리뷰 데이터셋
- 사용 컬럼: `Review`, `Rating`
- 라벨:
  - 0 = 부정
  - 1 = 긍정
  - 2 = 애매한 리뷰는 제외
- 클래스 불균형 완화: 긍정/부정 각각 최대 `MAX_PER_CLASS`개씩 균형 샘플링


In [15]:
# ============================================================
# 5-1. KR3 데이터셋 로드 및 train/test 생성
# ============================================================

def prepare_kr3_dataset():
    kr3 = load_dataset("leey4n/KR3", split="train")
    kr3_df = kr3.to_pandas()

    # 필요한 컬럼만 사용
    kr3_df = kr3_df[["Review", "Rating"]].rename(
        columns={"Review": "text", "Rating": "label"}
    )

    # 기본 전처리
    kr3_df = kr3_df.dropna(subset=["text", "label"])
    kr3_df["text"] = kr3_df["text"].astype(str).str.strip()
    kr3_df["label"] = kr3_df["label"].astype(int)
    kr3_df = kr3_df[kr3_df["text"].str.len() > 1]

    # 0=부정, 1=긍정만 사용. 2=애매한 리뷰 제외
    kr3_df = kr3_df[kr3_df["label"].isin([0, 1])].reset_index(drop=True)

    negative_df = kr3_df[kr3_df["label"] == 0]
    positive_df = kr3_df[kr3_df["label"] == 1]

    n_per_class = min(len(negative_df), len(positive_df), MAX_PER_CLASS)

    negative_sample = negative_df.sample(n=n_per_class, random_state=RANDOM_SEED)
    positive_sample = positive_df.sample(n=n_per_class, random_state=RANDOM_SEED)

    balanced_df = pd.concat([negative_sample, positive_sample], ignore_index=True)
    balanced_df = balanced_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    train_df, test_df = train_test_split(
        balanced_df,
        test_size=0.2,
        random_state=RANDOM_SEED,
        stratify=balanced_df["label"]
    )

    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print("KR3 전체 사용 데이터:", len(balanced_df))
    print("Train:", len(train_df), "Test:", len(test_df))
    print("Train label counts:")
    print(train_df["label"].value_counts())
    print("Test label counts:")
    print(test_df["label"].value_counts())

    return train_df, test_df


if TRAIN_KCBERT:
    train_df, test_df = prepare_kr3_dataset()
else:
    train_df, test_df = None, None

KR3 전체 사용 데이터: 4000
Train: 3200 Test: 800
Train label counts:
label
1    1600
0    1600
Name: count, dtype: int64
Test label counts:
label
1    400
0    400
Name: count, dtype: int64


In [16]:
# ============================================================
# 5-2. PyTorch Dataset 구성
# ============================================================

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [20]:
# ============================================================
# 5-3. KcBERT fine-tuning (수정됨: eval_strategy 및 Trainer 인자 대응)
# ============================================================

def train_kcbert_sentiment_model(train_df, test_df):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label={0: "부정", 1: "긍정"},
        label2id={"부정": 0, "긍정": 1}
    )

    train_dataset = ReviewDataset(
        train_df["text"],
        train_df["label"],
        tokenizer,
        max_length=MAX_LENGTH
    )

    test_dataset = ReviewDataset(
        test_df["text"],
        test_df["label"],
        tokenizer,
        max_length=MAX_LENGTH
    )

    training_args = TrainingArguments(
        output_dir=MODEL_SAVE_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.1,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=50,
        save_total_limit=1,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=RANDOM_SEED
    )

    # Trainer에서 tokenizer 인자를 제거하고 학습을 진행합니다.
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()

    eval_result = trainer.evaluate()
    print("Evaluation result:")
    print(eval_result)

    # 상세 report
    pred_output = trainer.predict(test_dataset)
    logits = pred_output.predictions
    y_true = pred_output.label_ids
    y_pred = np.argmax(logits, axis=1)

    print("\nClassification Report")
    print(classification_report(y_true, y_pred, target_names=["부정", "긍정"], zero_division=0))

    print("\nConfusion Matrix")
    print(confusion_matrix(y_true, y_pred))

    trainer.save_model(MODEL_SAVE_DIR)
    tokenizer.save_pretrained(MODEL_SAVE_DIR)

    return model, tokenizer


def load_kcbert_sentiment_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_SAVE_DIR)
    return model, tokenizer


if TRAIN_KCBERT:
    kcbert_model, kcbert_tokenizer = train_kcbert_sentiment_model(train_df, test_df)
else:
    kcbert_model, kcbert_tokenizer = load_kcbert_sentiment_model()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
kcbert_model.to(device)
kcbert_model.eval()

print("KcBERT model ready:", device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.282914,0.246450,0.905000,0.935484,0.870000,0.901554


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.282914,0.246450,1,0.905000,0.935484,0.870000,0.901554


Evaluation result:
{'eval_loss': 0.24644993245601654, 'eval_accuracy': 0.905, 'eval_precision': 0.9354838709677419, 'eval_recall': 0.87, 'eval_f1': 0.9015544041450777}



Classification Report
              precision    recall  f1-score   support

          부정       0.88      0.94      0.91       400
          긍정       0.94      0.87      0.90       400

    accuracy                           0.91       800
   macro avg       0.91      0.91      0.90       800
weighted avg       0.91      0.91      0.90       800


Confusion Matrix
[[376  24]
 [ 52 348]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KcBERT model ready: cuda


In [21]:
# ============================================================
# 6. KcBERT로 수집 리뷰 1차 감성분류
# ============================================================

@torch.no_grad()
def predict_sentiment_with_kcbert(texts, model, tokenizer, batch_size=32, max_length=MAX_LENGTH):
    results = []
    model.eval()

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = [str(t) for t in texts[start:start+batch_size]]

        enc = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )

        enc = {k: v.to(device) for k, v in enc.items()}

        outputs = model(**enc)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()

        for p in probs:
            negative_prob = float(p[0])
            positive_prob = float(p[1])

            # 2-class 모델을 3-class 서비스 라벨로 변환
            if positive_prob >= 0.55:
                sentiment_3class = "positive"
            elif positive_prob <= 0.35:
                sentiment_3class = "negative"
            else:
                sentiment_3class = "neutral"

            results.append({
                "negative_prob": negative_prob,
                "positive_prob": positive_prob,
                "sentiment_3class": sentiment_3class,
                "sentiment_binary": "positive" if positive_prob >= negative_prob else "negative"
            })

    return pd.DataFrame(results)


sentiment_df = predict_sentiment_with_kcbert(
    df["review_text"].tolist(),
    kcbert_model,
    kcbert_tokenizer,
    batch_size=32
)

tagged_df = pd.concat([df.reset_index(drop=True), sentiment_df.reset_index(drop=True)], axis=1)

print(tagged_df["sentiment_3class"].value_counts())
tagged_df.head()

100%|██████████| 1/1 [00:00<00:00, 32.40it/s]

sentiment_3class
positive    6
negative    6
Name: count, dtype: int64


,restaurant_name,review_text,menu_raw,review_date,recency_weight,menu_candidates,negative_prob,positive_prob,sentiment_3class,sentiment_binary
0,피자스쿨 수원성대점,친절하시고 진짜맛있어요,,2026-05-08,1.5,[],0.010530,0.989470,positive,positive
1,피자스쿨 수원성대점,"피자스쿨은 단순한 음식이 아니에요. 이건 사랑이고, 우정이고, 즐거움이에요! 뚱이랑...",,2025-12-18,1.0,[],0.351561,0.648439,positive,positive
2,피자스쿨 수원성대점,맛있게 먹었습니다~ 까르보나라 맛있네요,,2025-12-17,1.0,[],0.012772,0.987228,positive,positive
3,피자스쿨 수원성대점,점바점이 딱히 없는 괴자스쿨. 그러나 매장 영업은 비단 맛 뿐 만으로 판단되지는 않...,,2025-06-05,1.0,[],0.902824,0.097176,negative,negative
4,피자스쿨 수원성대점,태도가 뭐 피자 공짜로 나눠주기라도 하는줄ㅋㅋ,,2025-04-08,1.0,[],0.937783,0.062217,negative,negative


In [22]:
# ============================================================
# 7. 긍정 리뷰와 부정/주의 리뷰 분리
# ============================================================

# KcBERT 결과 기준:
# - positive: 긍정 리뷰
# - negative: 부정/주의 리뷰
# - neutral: 중립. 서비스 관점에서는 별도 보관하거나 최종 통계에만 반영
positive_df = tagged_df[tagged_df["sentiment_3class"] == "positive"].copy()
caution_df = tagged_df[tagged_df["sentiment_3class"] == "negative"].copy()
neutral_df = tagged_df[tagged_df["sentiment_3class"] == "neutral"].copy()

print(f"전체 리뷰: {len(tagged_df)}")
print(f"긍정 리뷰: {len(positive_df)}")
print(f"부정/주의 리뷰: {len(caution_df)}")
print(f"중립 리뷰: {len(neutral_df)}")

전체 리뷰: 12
긍정 리뷰: 6
부정/주의 리뷰: 6
중립 리뷰: 0


## 8. OpenAI Embedding 생성 및 클러스터링

KcBERT는 감성분류를 담당하고, 리뷰 의미 기반 그룹화는 embedding + KMeans가 담당합니다.

- 긍정 리뷰끼리 따로 클러스터링
- 부정/주의 리뷰끼리 따로 클러스터링


In [23]:
# ============================================================
# 8-1. OpenAI Embedding 생성
# ============================================================

def get_embeddings(texts, model=EMBEDDING_MODEL, batch_size=100):
    embeddings = []

    for start in tqdm(range(0, len(texts), batch_size)):
        batch = texts[start:start+batch_size]
        response = client.embeddings.create(
            model=model,
            input=batch
        )
        batch_embeddings = [item.embedding for item in response.data]
        embeddings.extend(batch_embeddings)

    return np.array(embeddings, dtype=np.float32)


def add_embeddings(df_part):
    if len(df_part) == 0:
        return df_part.copy(), np.empty((0, 0), dtype=np.float32)

    texts = df_part["review_text"].astype(str).tolist()
    emb = get_embeddings(texts)

    out = df_part.copy()
    out["embedding"] = list(emb)
    return out, emb


positive_df, positive_emb = add_embeddings(positive_df)
caution_df, caution_emb = add_embeddings(caution_df)

print("positive embedding shape:", positive_emb.shape)
print("caution embedding shape :", caution_emb.shape)

100%|██████████| 1/1 [00:01<00:00,  1.29s/it]

positive embedding shape: (6, 1536)
caution embedding shape : (6, 1536)


In [24]:
# ============================================================
# 8-2. KMeans 클러스터링
# ============================================================

def choose_best_k(embeddings, min_k=2, max_k=MAX_CLUSTERS):
    n = len(embeddings)

    if n < MIN_CLUSTER_SIZE:
        return 1

    max_k = min(max_k, n - 1)

    if max_k < min_k:
        return 1

    best_k = 1
    best_score = -1

    for k in range(min_k, max_k + 1):
        try:
            labels = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10).fit_predict(embeddings)

            if len(set(labels)) < 2:
                continue

            score = silhouette_score(embeddings, labels, metric="cosine")

            if score > best_score:
                best_score = score
                best_k = k

        except Exception:
            continue

    return best_k


def cluster_embedding_df(df_part, embeddings, group_name):
    out = df_part.copy()

    if len(out) == 0:
        out["cluster"] = []
        return out

    if len(out) < MIN_CLUSTER_SIZE:
        out["cluster"] = 0
        print(f"[{group_name}] 리뷰 수가 적어 단일 클러스터로 처리")
        return out

    k = choose_best_k(embeddings)

    if k == 1:
        out["cluster"] = 0
        print(f"[{group_name}] 단일 클러스터")
        return out

    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels = km.fit_predict(embeddings)

    out["cluster"] = labels

    print(f"[{group_name}] cluster 수: {k}")
    print(out["cluster"].value_counts().sort_index())

    return out


positive_clustered_df = cluster_embedding_df(positive_df, positive_emb, "positive")
caution_clustered_df = cluster_embedding_df(caution_df, caution_emb, "caution")

positive_clustered_df[["review_text", "sentiment_3class", "positive_prob", "cluster"]].head()

[positive] cluster 수: 3
cluster
0    3
1    2
2    1
Name: count, dtype: int64
[caution] cluster 수: 2
cluster
0    4
1    2
Name: count, dtype: int64


,review_text,sentiment_3class,positive_prob,cluster
0,친절하시고 진짜맛있어요,positive,0.989470,0
1,"피자스쿨은 단순한 음식이 아니에요. 이건 사랑이고, 우정이고, 즐거움이에요! 뚱이랑...",positive,0.648439,1
2,맛있게 먹었습니다~ 까르보나라 맛있네요,positive,0.987228,0
5,도이치바이트피자 맛있네요!,positive,0.988233,1
6,"최근 포장하러 방문했는데 굉장히 친절하셨음,,리뷰가 왜 이럴까 안타깝네요ㅠ",positive,0.817829,2


In [25]:
# ============================================================
# 9. 클러스터 대표 리뷰 추출
# ============================================================

def representative_reviews_by_cluster(clustered_df, embeddings, top_n=REPRESENTATIVE_REVIEWS_PER_CLUSTER):
    if len(clustered_df) == 0:
        return {}

    temp = clustered_df.reset_index(drop=True).copy()

    # embeddings 길이가 맞지 않으면 단순 head 사용
    if embeddings is None or len(embeddings) != len(temp):
        result = {}
        for cid, g in temp.groupby("cluster"):
            result[int(cid)] = g.head(top_n).to_dict("records")
        return result

    result = {}

    for cid in sorted(temp["cluster"].unique()):
        idxs = temp.index[temp["cluster"] == cid].tolist()
        cluster_emb = embeddings[idxs]

        if len(idxs) == 1:
            selected = temp.loc[idxs]
        else:
            centroid = cluster_emb.mean(axis=0, keepdims=True)
            dists = cosine_distances(cluster_emb, centroid).reshape(-1)

            part = temp.loc[idxs].copy()
            part["centroid_distance"] = dists

            # centroid에 가까운 대표성 + 최근 리뷰 가중치 반영
            part["represent_score"] = (
                -part["centroid_distance"]
                + 0.05 * part["recency_weight"].fillna(1.0)
            )

            selected = part.sort_values("represent_score", ascending=False).head(top_n)

        result[int(cid)] = selected.to_dict("records")

    return result


positive_reps = representative_reviews_by_cluster(
    positive_clustered_df,
    positive_emb if len(positive_emb) == len(positive_clustered_df) else None
)

caution_reps = representative_reviews_by_cluster(
    caution_clustered_df,
    caution_emb if len(caution_emb) == len(caution_clustered_df) else None
)

print("긍정 클러스터 대표 리뷰 수:", {k: len(v) for k, v in positive_reps.items()})
print("부정/주의 클러스터 대표 리뷰 수:", {k: len(v) for k, v in caution_reps.items()})

긍정 클러스터 대표 리뷰 수: {0: 3, 1: 2, 2: 1}
부정/주의 클러스터 대표 리뷰 수: {0: 4, 1: 2}


In [26]:
# ============================================================
# 10. LLM이 각 클러스터 의미 해석
# ============================================================

def safe_json_loads(text):
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return json.loads(text)


def compact_review_records(records):
    compact = []

    for r in records:
        compact.append({
            "review_text": r.get("review_text", ""),
            "menu_candidates": r.get("menu_candidates", []),
            "positive_prob": r.get("positive_prob", None),
            "negative_prob": r.get("negative_prob", None),
            "recency_weight": r.get("recency_weight", 1.0),
            "review_date": str(r.get("review_date", "")) if r.get("review_date", None) is not None else ""
        })

    return compact


def interpret_one_cluster(group_name, cluster_id, records):
    prompt = f"""당신은 음식점 리뷰 분석 전문가입니다.
아래는 {group_name} 리뷰 클러스터 {cluster_id}의 대표 리뷰입니다.

이 클러스터가 어떤 공통 주제를 의미하는지 해석하세요.
반드시 제공된 리뷰 근거만 사용하고, 없는 내용은 추측하지 마세요.

출력은 JSON만 사용하세요.
필드:
- cluster_id
- cluster_name
- summary
- key_aspects: 배열
- related_menus: 배열
- representative_phrases: 배열
- importance: high | medium | low
- evidence: 근거가 되는 리뷰 표현 요약

대표 리뷰:
{json.dumps(compact_review_records(records), ensure_ascii=False)}
"""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "You return valid JSON only."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        response_format={"type": "json_object"}
    )

    return safe_json_loads(response.choices[0].message.content)


def interpret_clusters_with_llm(rep_dict, group_name):
    interpretations = []

    for cid, records in rep_dict.items():
        if len(records) == 0:
            continue

        try:
            item = interpret_one_cluster(group_name, cid, records)
        except Exception as e:
            print(f"[WARN] cluster 해석 실패: {group_name} {cid}, error={e}")
            item = {
                "cluster_id": cid,
                "cluster_name": f"{group_name} cluster {cid}",
                "summary": "해석 실패",
                "key_aspects": [],
                "related_menus": [],
                "representative_phrases": [],
                "importance": "low",
                "evidence": ""
            }

        interpretations.append(item)
        time.sleep(0.2)

    return interpretations


positive_cluster_interpretations = interpret_clusters_with_llm(positive_reps, "긍정")
caution_cluster_interpretations = interpret_clusters_with_llm(caution_reps, "부정/주의")

print("[긍정 클러스터 해석]")
print(json.dumps(positive_cluster_interpretations, ensure_ascii=False, indent=2))

print("\n[부정/주의 클러스터 해석]")
print(json.dumps(caution_cluster_interpretations, ensure_ascii=False, indent=2))

[긍정 클러스터 해석]
[
  {
    "cluster_id": 0,
    "cluster_name": "맛과 친절",
    "summary": "이 클러스터는 음식의 맛과 직원의 친절함에 대한 긍정적인 경험을 공유합니다.",
    "key_aspects": [
      "음식의 맛",
      "직원 친절"
    ],
    "related_menus": [
      "까르보나라"
    ],
    "representative_phrases": [
      "맛 짱",
      "친절하시고 진짜맛있어요",
      "맛있게 먹었습니다~ 까르보나라 맛있네요"
    ],
    "importance": "high",
    "evidence": "리뷰에서 음식의 맛이 뛰어나고 직원들이 친절하다는 긍정적인 표현이 반복적으로 나타남."
  },
  {
    "cluster_id": 1,
    "cluster_name": "사랑과 즐거움의 피자 경험",
    "summary": "이 클러스터는 피자를 단순한 음식 이상의 경험으로 여기는 리뷰들로 구성되어 있습니다. 피자는 사랑, 우정, 즐거움과 연결되어 있으며, 함께 나누는 순간이 특별하다는 메시지를 전달합니다.",
    "key_aspects": [
      "사랑",
      "우정",
      "즐거움",
      "공유 경험"
    ],
    "related_menus": [],
    "representative_phrases": [
      "이건 사랑이고, 우정이고, 즐거움이에요!",
      "세상을 다 가진 기분이 들죠!",
      "맛있네요!"
    ],
    "importance": "high",
    "evidence": "리뷰에서 피자를 단순한 음식이 아닌 사랑과 즐거움의 상징으로 묘사하며, 함께 나누는 경험의 중요성을 강조하고 있습니다."
  },
  {
    "cluster_id": 2,
    "cluster_name": "친절한 서비

In [37]:
import numpy as np

def build_menu_long_df(df_input):
    rows = []
    for _, row in df_input.iterrows():
        menus = row.get("menu_candidates", [])
        for menu in (menus if isinstance(menus, list) else []):
            rows.append({
                "menu": menu,
                "sentiment_3class": row.get("sentiment_3class", "neutral"),
                "positive_prob": row.get("positive_prob", 0.5),
                "review_text": row.get("review_text", ""),
                "recency_weight": row.get("recency_weight", 1.0)
            })
    return pd.DataFrame(rows)

tagged_df["menu_candidates"] = df["menu_candidates"]
menu_long_df = build_menu_long_df(tagged_df)

if not menu_long_df.empty:
    menu_summary = (
        menu_long_df
        .assign(
            positive_w=lambda x: np.where(x["sentiment_3class"] == "positive", x["recency_weight"], 0.0),
            caution_w=lambda x: np.where(x["sentiment_3class"] == "negative", x["recency_weight"], 0.0),
            neutral_w=lambda x: np.where(x["sentiment_3class"] == "neutral", x["recency_weight"], 0.0)
        )
        .groupby("menu")
        .agg(
            mention_count=("review_text", "count"),
            positive_count=("sentiment_3class", lambda s: (s == "positive").sum()),
            caution_count=("sentiment_3class", lambda s: (s == "negative").sum()),
            neutral_count=("sentiment_3class", lambda s: (s == "neutral").sum()),
            positive_weighted=("positive_w", "sum"),
            caution_weighted=("caution_w", "sum"),
            total_weighted=("recency_weight", "sum"),
            avg_positive_prob=("positive_prob", "mean")
        )
        .reset_index()
    )

    menu_summary["positive_ratio_weighted"] = menu_summary["positive_weighted"] / menu_summary["total_weighted"]
    menu_summary["caution_ratio_weighted"] = menu_summary["caution_weighted"] / menu_summary["total_weighted"]
    menu_summary = menu_summary.sort_values("mention_count", ascending=False)
    display(menu_summary.head(10))
else:
    print("데이터가 비어있습니다. 4번 셀부터 다시 실행해주세요.")

,menu,mention_count,positive_count,caution_count,neutral_count,positive_weighted,caution_weighted,total_weighted,avg_positive_prob,positive_ratio_weighted,caution_ratio_weighted
3,피자,6,2,4,0,2.0,4.0,6.0,0.307145,0.333333,0.666667
0,까르보나라,1,1,0,0,1.0,0.0,1.0,0.987228,1.000000,0.000000
1,도이치바이트,1,1,0,0,1.0,0.0,1.0,0.988233,1.000000,0.000000
2,치즈피자,1,0,1,0,0.0,1.0,1.0,0.026081,0.000000,1.000000


In [38]:
# ============================================================
# 12. 전체 통계 및 최근 리뷰 흐름
# ============================================================

def make_overall_stats(tagged_df):
    total = len(tagged_df)
    counts = tagged_df["sentiment_3class"].value_counts().to_dict()

    weighted = tagged_df.groupby("sentiment_3class")["recency_weight"].sum().to_dict()
    total_w = tagged_df["recency_weight"].sum()

    stats = {
        "total_reviews": total,
        "sentiment_counts": counts,
        "weighted_sentiment": weighted,
        "positive_ratio": counts.get("positive", 0) / total if total else 0.0,
        "negative_ratio": counts.get("negative", 0) / total if total else 0.0,
        "neutral_ratio": counts.get("neutral", 0) / total if total else 0.0,
        "weighted_positive_ratio": weighted.get("positive", 0.0) / total_w if total_w else 0.0,
        "weighted_negative_ratio": weighted.get("negative", 0.0) / total_w if total_w else 0.0,
        "weighted_neutral_ratio": weighted.get("neutral", 0.0) / total_w if total_w else 0.0,
        "avg_positive_prob": float(tagged_df["positive_prob"].mean()) if total else 0.0,
        "date_min": str(tagged_df["review_date"].min()) if not tagged_df["review_date"].isna().all() else None,
        "date_max": str(tagged_df["review_date"].max()) if not tagged_df["review_date"].isna().all() else None
    }

    return stats


def build_recent_review_summary(tagged_df, top_n=10):
    if tagged_df["review_date"].isna().all():
        recent = tagged_df.sort_values("recency_weight", ascending=False).head(top_n)
    else:
        recent = tagged_df.sort_values(["review_date", "recency_weight"], ascending=[False, False]).head(top_n)

    return recent[[
        "review_date", "review_text", "sentiment_3class",
        "positive_prob", "negative_prob", "recency_weight", "menu_candidates"
    ]].to_dict("records")


overall_stats = make_overall_stats(tagged_df)
recent_reviews = build_recent_review_summary(tagged_df)

print(json.dumps(overall_stats, ensure_ascii=False, indent=2))
print(pd.DataFrame(recent_reviews).head())

{
  "total_reviews": 12,
  "sentiment_counts": {
    "positive": 6,
    "negative": 6
  },
  "weighted_sentiment": {
    "negative": 6.0,
    "positive": 6.5
  },
  "positive_ratio": 0.5,
  "negative_ratio": 0.5,
  "neutral_ratio": 0.0,
  "weighted_positive_ratio": 0.52,
  "weighted_negative_ratio": 0.48,
  "weighted_neutral_ratio": 0.0,
  "avg_positive_prob": 0.4701481826292972,
  "date_min": "2018-09-30 00:00:00",
  "date_max": "2026-05-08 00:00:00"
}
  review_date                                        review_text  \
0  2026-05-08                                       친절하시고 진짜맛있어요   
1  2025-12-18  피자스쿨은 단순한 음식이 아니에요. 이건 사랑이고, 우정이고, 즐거움이에요! 뚱이랑...   
2  2025-12-17                              맛있게 먹었습니다~ 까르보나라 맛있네요   
3  2025-06-05  점바점이 딱히 없는 괴자스쿨. 그러나 매장 영업은 비단 맛 뿐 만으로 판단되지는 않...   
4  2025-04-08                          태도가 뭐 피자 공짜로 나눠주기라도 하는줄ㅋㅋ   

  sentiment_3class  positive_prob  negative_prob  recency_weight  \
0         positive       0.989470       0.010530             1.5 

In [39]:
# ============================================================
# 13. 최종 음식점 AI 프로필 생성
# ============================================================

def make_cluster_context(interpretations):
    if not interpretations:
        return "[]"
    return json.dumps(interpretations, ensure_ascii=False, indent=2)


def make_menu_context(menu_summary, top_n=15):
    if len(menu_summary) == 0:
        return "메뉴별 데이터 없음"

    cols = [
        "menu", "mention_count", "positive_count", "caution_count", "neutral_count",
        "positive_ratio_weighted", "caution_ratio_weighted", "avg_positive_prob"
    ]

    return menu_summary[cols].head(top_n).to_json(orient="records", force_ascii=False)


def generate_final_profile(
    tagged_df,
    positive_cluster_interpretations,
    caution_cluster_interpretations,
    menu_summary,
    recent_reviews
):
    store_name = tagged_df["restaurant_name"].iloc[0] if len(tagged_df) else "음식점"
    stats = make_overall_stats(tagged_df)

    prompt = f"""당신은 음식점 리뷰 분석 AI입니다.
아래 분석 결과만 근거로 최종 음식점 AI 프로필을 생성하세요.
데이터에 없는 내용은 추측하지 마세요.

중요:
- 감성분류는 KcBERT 기반 모델 결과입니다.
- LLM은 이 결과를 재분류하지 말고, 클러스터와 메뉴 반응을 해석하는 데 집중하세요.
- 최근 리뷰 가중치가 반영된 비율을 함께 고려하세요.

음식점명: {store_name}

[전체 통계]
{json.dumps(stats, ensure_ascii=False, indent=2)}

[긍정 클러스터 해석]
{make_cluster_context(positive_cluster_interpretations)}

[부정/주의 클러스터 해석]
{make_cluster_context(caution_cluster_interpretations)}

[메뉴별 반응: 최근 리뷰 가중치 반영]
{make_menu_context(menu_summary)}

[최근 리뷰 예시]
{json.dumps(recent_reviews, ensure_ascii=False, default=str, indent=2)}

출력은 JSON만 사용하세요.
필드:
- store_name
- overall_score: 1~5 숫자
- one_line_summary
- main_strengths: 배열
- main_weaknesses: 배열
- caution_points: 배열
- recommended_menus: 배열. 각 원소는 menu, reason, confidence 포함
- menu_reactions: 배열. 각 원소는 menu, positive_reaction, negative_or_caution_reaction 포함
- recent_trend
- recommended_customer
- not_recommended_customer
- final_summary
"""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "You return valid JSON only."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        response_format={"type": "json_object"}
    )

    return safe_json_loads(response.choices[0].message.content)


final_profile = generate_final_profile(
    tagged_df,
    positive_cluster_interpretations,
    caution_cluster_interpretations,
    menu_summary,
    recent_reviews
)

print(json.dumps(final_profile, ensure_ascii=False, indent=2))

{
  "store_name": "피자스쿨 수원성대점",
  "overall_score": 3,
  "one_line_summary": "맛과 친절함이 돋보이나, 서비스와 음식 질에 대한 불만이 존재하는 피자 전문점.",
  "main_strengths": [
    "음식의 맛",
    "직원 친절",
    "피자를 통한 특별한 경험"
  ],
  "main_weaknesses": [
    "서비스 태도",
    "음식의 질",
    "토핑의 양"
  ],
  "caution_points": [
    "불친절한 서비스",
    "음식의 질에 대한 불만"
  ],
  "recommended_menus": [
    {
      "menu": "까르보나라",
      "reason": "리뷰에서 매우 긍정적인 반응을 얻음.",
      "confidence": 0.987
    },
    {
      "menu": "도이치바이트",
      "reason": "리뷰에서 긍정적인 반응을 얻음.",
      "confidence": 0.988
    }
  ],
  "menu_reactions": [
    {
      "menu": "피자",
      "positive_reaction": 2,
      "negative_or_caution_reaction": 4
    },
    {
      "menu": "까르보나라",
      "positive_reaction": 1,
      "negative_or_caution_reaction": 0
    },
    {
      "menu": "도이치바이트",
      "positive_reaction": 1,
      "negative_or_caution_reaction": 0
    },
    {
      "menu": "치즈피자",
      "positive_reaction": 0,
      "negative_or_caution_reaction": 1
    }
 

In [40]:
# ============================================================
# 14. 결과 저장
# ============================================================

tagged_df.to_csv("01_kcbert_tagged_reviews.csv", index=False, encoding="utf-8-sig")

positive_clustered_df.drop(columns=["embedding"], errors="ignore").to_csv(
    "02_positive_clustered_reviews.csv",
    index=False,
    encoding="utf-8-sig"
)

caution_clustered_df.drop(columns=["embedding"], errors="ignore").to_csv(
    "03_caution_clustered_reviews.csv",
    index=False,
    encoding="utf-8-sig"
)

menu_summary.to_csv(
    "04_menu_reaction_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

with open("05_positive_cluster_interpretations.json", "w", encoding="utf-8") as f:
    json.dump(positive_cluster_interpretations, f, ensure_ascii=False, indent=2)

with open("06_caution_cluster_interpretations.json", "w", encoding="utf-8") as f:
    json.dump(caution_cluster_interpretations, f, ensure_ascii=False, indent=2)

with open("07_final_restaurant_ai_profile.json", "w", encoding="utf-8") as f:
    json.dump(final_profile, f, ensure_ascii=False, indent=2)

print("저장 완료")
print("- 01_kcbert_tagged_reviews.csv")
print("- 02_positive_clustered_reviews.csv")
print("- 03_caution_clustered_reviews.csv")
print("- 04_menu_reaction_summary.csv")
print("- 05_positive_cluster_interpretations.json")
print("- 06_caution_cluster_interpretations.json")
print("- 07_final_restaurant_ai_profile.json")

저장 완료
- 01_kcbert_tagged_reviews.csv
- 02_positive_clustered_reviews.csv
- 03_caution_clustered_reviews.csv
- 04_menu_reaction_summary.csv
- 05_positive_cluster_interpretations.json
- 06_caution_cluster_interpretations.json
- 07_final_restaurant_ai_profile.json


In [41]:
# ============================================================
# 15. 발표/확인용 출력
# ============================================================

def print_profile(profile):
    print("=" * 70)
    print(f"[AI Profile] {profile.get('store_name', '')}")
    print("=" * 70)
    print(f"Overall Score: {profile.get('overall_score', '')}")
    print(f"Summary: {profile.get('one_line_summary', '')}")
    print()

    print("[Main Strengths]")
    for x in profile.get("main_strengths", []):
        print("-", x)

    print("\n[Main Weaknesses]")
    for x in profile.get("main_weaknesses", []):
        print("-", x)

    print("\n[Caution Points]")
    for x in profile.get("caution_points", []):
        print("-", x)

    print("\n[Recommended Menus]")
    for x in profile.get("recommended_menus", []):
        if isinstance(x, dict):
            print(f"- {x.get('menu')}: {x.get('reason')} ({x.get('confidence')})")
        else:
            print("-", x)

    print("\n[Menu Reactions]")
    for x in profile.get("menu_reactions", []):
        if isinstance(x, dict):
            print(f"- {x.get('menu')}: + {x.get('positive_reaction')} / ! {x.get('negative_or_caution_reaction')}")
        else:
            print("-", x)

    print("\n[Recent Trend]")
    print(profile.get("recent_trend", ""))

    print("\n[Recommended Customer]")
    print(profile.get("recommended_customer", ""))

    print("\n[Not Recommended Customer]")
    print(profile.get("not_recommended_customer", ""))

    print("\n[Final Summary]")
    print(profile.get("final_summary", ""))


print_profile(final_profile)

[AI Profile] 피자스쿨 수원성대점
Overall Score: 3
Summary: 맛과 친절함이 돋보이나, 서비스와 음식 질에 대한 불만이 존재하는 피자 전문점.

[Main Strengths]
- 음식의 맛
- 직원 친절
- 피자를 통한 특별한 경험

[Main Weaknesses]
- 서비스 태도
- 음식의 질
- 토핑의 양

[Caution Points]
- 불친절한 서비스
- 음식의 질에 대한 불만

[Recommended Menus]
- 까르보나라: 리뷰에서 매우 긍정적인 반응을 얻음. (0.987)
- 도이치바이트: 리뷰에서 긍정적인 반응을 얻음. (0.988)

[Menu Reactions]
- 피자: + 2 / ! 4
- 까르보나라: + 1 / ! 0
- 도이치바이트: + 1 / ! 0
- 치즈피자: + 0 / ! 1

[Recent Trend]
최근 리뷰에서 긍정적인 경험이 다수 나타나지만, 서비스와 음식 질에 대한 불만도 여전히 존재.

[Recommended Customer]
피자와 까르보나라를 좋아하는 고객

[Not Recommended Customer]
서비스 품질을 중시하는 고객

[Final Summary]
피자스쿨 수원성대점은 맛과 친절함이 돋보이는 곳이지만, 서비스와 음식 질에 대한 불만이 있어 주의가 필요합니다. 특히 까르보나라와 도이치바이트는 추천할 만한 메뉴입니다.


In [42]:
from google.colab import files

# 다운로드할 주요 결과 파일 리스트
files_to_download = [
    "07_final_restaurant_ai_profile.json",
    "01_kcbert_tagged_reviews.csv",
    "04_menu_reaction_summary.csv"
]

for file_name in files_to_download:
    if os.path.exists(file_name):
        files.download(file_name)
    else:
        print(f"파일을 찾을 수 없습니다: {file_name}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
import shutil
from google.colab import files

# 1. 모델 폴더 압축 (팀원 전달용)
model_path = './kcbert_kr3_sentiment_model'
zip_path = './kcbert_sentiment_model'
if os.path.exists(model_path):
    shutil.make_archive(zip_path, 'zip', model_path)
    print(f"모델 압축 완료: {zip_path}.zip")

# 2. 다운로드할 파일 리스트 구성
final_delivery_files = [
    "kcbert_sentiment_model.zip",
    "07_final_restaurant_ai_profile.json",
    "01_kcbert_tagged_reviews.csv",
    "04_menu_reaction_summary.csv"
]

# 3. 파일 순차 다운로드
for f_name in final_delivery_files:
    if os.path.exists(f_name):
        print(f"{f_name} 다운로드를 시작합니다...")
        files.download(f_name)
    else:
        print(f"파일을 찾을 수 없습니다: {f_name}")

모델 압축 완료: ./kcbert_sentiment_model.zip
kcbert_sentiment_model.zip 다운로드를 시작합니다...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

07_final_restaurant_ai_profile.json 다운로드를 시작합니다...


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

04_menu_reaction_summary.csv 다운로드를 시작합니다...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>